# Домашнее задание 3. Ранжирование (20p)

В ДЗ 2 мы строили retrieval-этап рекомендательной системы. Теперь добавим
второй этап — ranking, тем самым формируя полноценный рекомендательный пайплайн.

Итак, пусть каждый кандидатогенератор возвращает не более 100 товаров. 
Кандидатогенераторов у нас несколько, и их необходимо объединить посредством ранкинга.
Финальная выдача будет состоять из 20 айтемов.

В качестве кандидатогенераторов возьмем TopPopular, ContentItemKNN и ALS из предыдущей работы,
а в качестве ранкера будет CatBoost. Попробуем две вариации - CatBoostClassifier и CatBoostRanker. 
Качество итоговых рекомендаций будем оценивать с помощью `ExpNDCG`, `HitRate` и `MRR`.

В рамках работы необходимо:

1. собрать point-in-time датамарт `Raw → Aggs → Features` с историческими признаками пользователей и товаров;
2. получить кандидатов из нескольких retrieval-источников, объединить их, дедуплицировать, разметить и дополнить признаками;
3. обучить CatBoostClassifier и CatBoostRanker, подобрать гиперпараметры с помощью `optuna` и выбрать лучший вариант по качеству на `validation`;
4. оценить итоговую связку retrieval + ranking на `test`.

Реализации retrieval-моделей можно перенести из ДЗ 2 без изменений.

In [ ]:
import hashlib
import math
import random
from abc import ABC, abstractmethod
from collections import Counter
from collections.abc import Callable, Iterable, Mapping, Sequence
from datetime import date, datetime, timedelta
from pathlib import Path
from typing import Any, Literal, Self

import implicit
import matplotlib.pyplot as plt
import numpy as np
import optuna
import polars as pl
import tests
from catboost import CatBoostClassifier, CatBoostRanker, Pool
from numpy.typing import NDArray
from optuna.samplers import TPESampler
from scipy import sparse
from sentence_transformers import SentenceTransformer
from sklearn.preprocessing import normalize
from tqdm.auto import tqdm
from IPython.display import display

SEED = 42
RETRIEVAL_K = 100
EVAL_K = 20
DATA_DIR = Path("../data")
DATAMART_DIR = DATA_DIR / "ranking_datamart"

random.seed(SEED)
np.random.seed(SEED)
optuna.logging.set_verbosity(optuna.logging.WARNING)

interactions = pl.read_parquet(DATA_DIR / "interactions.parquet")
items = pl.read_parquet(DATA_DIR / "items.parquet")
users = pl.read_parquet(DATA_DIR / "users.parquet")

interactions = interactions.with_columns(
    pl.col("user_id").cast(pl.Utf8),
    pl.col("item_id").cast(pl.Utf8),
    pl.col("rating").cast(pl.Float64),
    pl.col("timestamp").cast(pl.Datetime("us", "UTC")),
)

users = users.with_columns(pl.col("user_id").cast(pl.Utf8)).filter(
    pl.col("user_id").is_in(interactions["user_id"].unique().to_list())
)
items = items.with_columns(pl.col("item_id").cast(pl.Utf8)).filter(
    pl.col("item_id").is_in(interactions["item_id"].unique().to_list())
)

## 1. Предобработка (1p)

Будем решать задачу рекомендаций с недельным горизонтом: на начало каждой недели для пользователя нужно сформировать список товаров, а события этой недели будут использоваться как ground truth. Поэтому объединим взаимодействия в сессии по паре `(user_id, week_start)`. В предыдущем ДЗ наблюдаемая оценка считалась фактом взаимодействия. Сохраним эту трактовку: само наличие оценки — наблюдаемое событие, а число от 1 до 5 отражает его силу.

Для пользователя $u$ и календарной недели, начинающейся в момент $t$, определим запрос ранжирования как

$$s=(u,t).$$

Здесь $t$ — понедельник, 00:00 UTC.

Нужно реализовать функцию `add_weekly_sessions`: нормализовать timestamp к началу недели и собрать `session_id` из даты начала недели и `user_id`.

In [ ]:
def add_weekly_sessions(interactions: pl.DataFrame) -> pl.DataFrame:
    """Добавляет в данные колонки `week_start` и `session_id`."""
    # PUT YOUR CODE HERE


tests.check_weekly_sessions(add_weekly_sessions)


Посмотрим на размер сессий.

In [ ]:
interactions = add_weekly_sessions(interactions)
session_sizes = interactions.group_by("session_id").agg(pl.len().alias("events"))

plt.figure(figsize=(8, 3))
plt.hist(session_sizes["events"].to_numpy(), bins=30)
plt.xlabel("Событий в недельной сессии")
plt.ylabel("Число сессий")
plt.title("Распределение длины сессий")
plt.show()

## 2. Глобальное временное разбиение (1p)

Случайное разбиение здесь использовать нельзя: оно позволило бы модели учиться на событиях, произошедших позже оцениваемой недели. Поэтому разделим данные по времени: на `train` обучим модели, по `validation` выберем гиперпараметры, а `test` оставим только для финальной оценки.

Границы задаются сразу для всех пользователей и проходят между неделями. Благодаря этому каждая недельная сессия целиком относится только к одной выборке.

Выберем две границы $T_{val}$ и $T_{test}$, приходящиеся на понедельник в 00:00 UTC:

$$Train=\{e:t_e<T_{val}\},$$
$$Validation=\{e:T_{val}\le t_e<T_{test}\},$$
$$Test=\{e:t_e\ge T_{test}\}.$$

Границы допустимы только между полными неделями. Доли `train_ratio`, `validation_ratio` и `test_ratio` относятся к числу событий, а не к числу недель. Из всех допустимых пар границ выберите ту, для которой накопленные доли событий ближе всего к заданным; при равенстве используйте более ранние границы. Необходимо реализовать функцию `temporal_week_split`, которая выберет глобальные недельные границы и вернёт три непересекающиеся части данных. Внутри периода нужно придерживаться исходного порядка событий.

In [ ]:
def temporal_week_split(
    interactions: pl.DataFrame,
    train_ratio: float = 0.8,
    validation_ratio: float = 0.1,
    test_ratio: float = 0.1,
) -> tuple[pl.DataFrame, pl.DataFrame, pl.DataFrame]:
    """Разделяет данные на train, validation и test по границам недель."""
    ratios = (train_ratio, validation_ratio, test_ratio)
    if any(r <= 0 for r in ratios) or not np.isclose(sum(ratios), 1.0):
        raise ValueError("Все доли должны быть положительными и суммироваться в 1")

    # PUT YOUR CODE HERE
    return


tests.check_temporal_week_split(add_weekly_sessions, temporal_week_split)

In [ ]:
train_events, val_events, test_events = temporal_week_split(interactions)

t_val = val_events["week_start"].min()
t_test = test_events["week_start"].min()

split_stats = pl.DataFrame(
    {
        "split": ["train", "validation", "test"],
        "events": [len(train_events), len(val_events), len(test_events)],
        "sessions": [
            x["session_id"].n_unique() for x in (train_events, val_events, test_events)
        ],
        "users": [
            x["user_id"].n_unique() for x in (train_events, val_events, test_events)
        ],
        "items": [
            x["item_id"].n_unique() for x in (train_events, val_events, test_events)
        ],
    }
)

display(split_stats)
print("T_val:", t_val, "T_test:", t_test)

Зафиксируем главное правило для всей домашней работы. В начале недели $t$ известны только события с моментом времени **строго раньше** её начала:

$$H(t)=\{e:timestamp_e<t\}.$$

Это касается негативного сэмплирования, популярности товаров и всех признаков. Предположим, пользователь поставил оценку в среду. Эту оценку разрешено использовать как целевой ответ для текущей недели, но **нельзя** включать в фичи на её начало.

## 3. Базис (1p)

Обучающую таблицу удобно разделить на `basis` и признаки. `Basis` определяет состав строк и target: какая пара `(session_id, item_id)` присутствует в выборке и насколько товар релевантен. Признаки будут рассчитаны отдельно по прошлым событиям и присоединены к этим строкам позднее. 

Иными словами, `basis` отвечает на вопрос «что ранжируем и с какой меткой», а features — «что было известно об этих объектах на момент формирования рекомендаций». Итоговый ranking dataset получается point-in-time объединением этих двух частей.

Сначала построим положительную часть `basis` — товары, с которыми пользователь впервые взаимодействовал в данной недельной сессии. После генерации кандидатов добавим к ней псевдонегативы, так как истинных негативов в данных нет.

Все оценки 1–5 — наблюдаемые положительные взаимодействия. Для classifier
используем `binary_target`, для ranker сохраняем исходную `relevance` 1–5.

Повторно рекомендовать уже оценённый товар не будем. Поэтому оставим только первую неделю
знакомства пользователя с товаром; несколько оценок внутри этой недели схлопываем через `max(rating)`.

Реализуйте функцию `build_positive_labels`, которая соберёт разметку положительных примеров. Выход — таблица с колонками `session_id`, `week_start`, `user_id`, `item_id`, `binary_target` (всегда `1`) и `relevance` (1–5), по одной строке на пару `(session_id, item_id)`.

In [ ]:
def build_positive_labels(interactions: pl.DataFrame) -> pl.DataFrame:
    """Строит ground truth взаимодействия user–item."""
    # PUT YOUR CODE HERE


tests.check_positive_labels(add_weekly_sessions, build_positive_labels)


Позднее для каждого candidate pool выполним left join с полным ground truth
соответствующей недели:

| Кандидат | `binary_target` | `relevance` |
|---|---:|---:|
| наблюдался в этой недельной сессии | 1 | 1–5 |
| не наблюдался | 0 | 0 |

Это всё ещё псевдонегативы: отсутствие оценки не доказывает неприязнь. Но такие
негативы согласованы с реальной задачей ранкера — переставить товары, уже
отобранные retrieval-моделями.

In [ ]:
positives = build_positive_labels(interactions)
val_truth = positives.filter(
    (pl.col("week_start") >= t_val) & (pl.col("week_start") < t_test)
)
test_truth = positives.filter(pl.col("week_start") >= t_test)

label_dist = positives.group_by("relevance").agg(pl.len().alias("n")).sort("relevance")
plt.figure(figsize=(7, 3))
plt.bar(label_dist["relevance"].to_list(), label_dist["n"].to_list())
plt.xlabel("Observed relevance")
plt.ylabel("Число first-time пар")
plt.title("Распределение положительных оценок")
plt.show()

## 4. Кастомный датамарт (6p)

Датамарт — это подготовленный под конкретную задачу слой данных между исходными событиями и моделью. В отличие от общей таблицы взаимодействий, он хранит данные в форме, из которой можно воспроизводимо собирать обучающие выборки и признаки для нужного момента времени.

В этой работе датамарт позволит один раз сохранить сырые события и промежуточные агрегаты, а затем использовать их при построении признаков для разных недель. Благодаря этому нам не придётся заново обрабатывать всю историю для каждого эксперимента, а единое правило временного среза поможет не допустить утечку будущих событий в обучение.

Построим простой датамарт, состоящий из трёх слоёв:

1. **Raw** — исходные события по неделям.
2. **Aggs** — недельные счётчики по паре пользователь–товар.
3. **Features** — готовые исторические признаки на начало недели.

### 4.1. Raw: разложим события по неделям (1p)

Путь `raw/events/week=YYYY-MM-DD/events.parquet` соответствует событиям за интервал $[t,t+7\text{ дней})$. Напишите функцию `build_raw_layer`, сохраняющую каждое взаимодействие ровно в одну недельную партицию и не удаляющую повторные оценки.

In [ ]:
def build_raw_layer(interactions: pl.DataFrame, datamart_dir: Path) -> None:
    """Разбивает события по неделям и записывает каждую часть в Raw layer."""
    # PUT YOUR CODE HERE


tests.check_raw_layer(add_weekly_sessions, build_raw_layer)


### 4.2. Aggs: посчитаем недельные счётчики (2p)

Следующий слой содержит по одной строке на `(week_start, user_id, item_id)` и позволяет собрать исторические статистики без повторного чтения сырых событий.

Для пары $(u,i)$ и недели $w$ нужны:

- `n_events` — число событий;
- `rating_sum` — сумма оценок, `rating_count` — число оценок;
- `n_high_ratings` — число оценок 4–5;
- `first_event_ts` и `last_event_ts` — границы наблюдаемой активности.

Недельные счётчики посчитайте в функции `calculate_weekly_counters`.

In [ ]:
def calculate_weekly_counters(events: pl.DataFrame) -> pl.DataFrame:
    """Агрегирует события до одной строки на `(week, user, item)`."""
    # PUT YOUR CODE HERE
    return


tests.check_weekly_counters(add_weekly_sessions, calculate_weekly_counters)


Далее необходимо реализовать `build_aggs_layer`: функция читает недельные raw-партиции, вызывает расчёт счётчиков и сохраняет `aggs/user_item/week=.../counters.parquet`.

In [ ]:
def build_aggs_layer(datamart_dir: Path) -> None:
    """Читает недельные Raw-партиции и записывает агрегаты user–item."""
    # PUT YOUR CODE HERE


tests.check_aggs_layer(add_weekly_sessions, build_raw_layer, build_aggs_layer)


Теперь реализуйте `read_weekly_counters(datamart_dir, week_start, lookback_weeks=None)`: функция читает только партиции из окна $[week\_start - lookback\_weeks, week\_start)$, а при `lookback_weeks=None` — всю историю строго до `week_start`. При `lookback_weeks <= 0` функция выбрасывает `ValueError`, а строки с `week_start >= week_start` в результат не попадают. Вся сборка признаков должна опираться на Aggs, а не заново сканировать Raw.

In [ ]:
def read_weekly_counters(
    datamart_dir: Path,
    week_start: datetime,
    lookback_weeks: int | None = None,
) -> pl.DataFrame:
    """Читает только партиции Aggs из окна `[week_start - lookback, week_start)`."""
    if lookback_weeks is not None and lookback_weeks <= 0:
        raise ValueError("lookback_weeks должен быть положительным")

    paths = sorted(
        (datamart_dir / "aggs" / "user_item").glob("week=*/counters.parquet")
    )
    if not paths:
        raise FileNotFoundError("Сначала постройте aggs layer")

    # PUT YOUR CODE HERE
    return


tests.check_weekly_counters_reader(
    add_weekly_sessions, build_raw_layer, build_aggs_layer, read_weekly_counters
)

### 4.3. Features (3p)

Мы хотим различать общую популярность товара и недавний интерес к нему. Поэтому для расчета признаков используем два временных окна на начало недели $t$:

$$W_{lifetime}(t)=(-\infty,t),$$
$$W_{4w}(t)=[t-28\text{ дней},t).$$

Для каждого временного среза будем получать признаки пользователя, товара, общие статистики и при доступных категориях признаки `user × category`.

Начнем с категорий. В метаданных Amazon категория может отсутствовать, быть строкой или списком категорий. Зафиксируем одну детерминированную категорию товара, если она присутствует, и сохраним словарь категорий отдельно. Логика следующая:

- Источник — колонка `category`, если она есть, иначе `categories`. Если обе колонки отсутствуют, функция возвращает `None` и не создаёт файл. Значение приводим к списку строк: `null` — пустой список, строка — список из одного элемента. Каждую метку очищаем от пробелов по краям, пустые строки и повторы внутри товара отбрасываем.
- Частота метки — число товаров каталога, у которых она есть.
- Выбор: самая редкая метка товара, при равенстве — лексикографически меньшая. Порядок строк в `items` на результат не влияет.
- Товар без меток получает `"unknown"`.
- Выход — таблица `item_id`/`category` (обе `String`), по строке на каждый товар, сохранённая в `common/item_categories.parquet` внутри `datamart_dir`.

Эту логику реализуйте в функции `prepare_item_categories(items, datamart_dir)`.

In [ ]:
def prepare_item_categories(
    items: pl.DataFrame, datamart_dir: Path
) -> pl.DataFrame | None:
    """Строит таблицу соответствий между товарами и категориями."""
    source_column = next(
        (column for column in ("category", "categories") if column in items.columns),
        None,
    )
    if source_column is None:
        return None

    # PUT YOUR CODE HERE
    return


tests.check_item_categories(prepare_item_categories)

Обозначим через $N_u(t)$ число исторических оценок пользователя, через $S_u(t)$ их сумму, а через $H_u(t)$ число оценок 4–5. Тогда

$$user\_mean\_rating_u(t)=\frac{S_u(t)}{N_u(t)},\qquad user\_high\_rating\_share_u(t)=\frac{H_u(t)}{N_u(t)}.$$

По товарам считаем число событий, число уникальных пользователей, среднюю оценку и недавнюю популярность. Чтобы не доверять слишком сильно среднему рейтингу товара с одной оценкой, используем байесовское сглаживание:

$$BayesRating_i(t)=\frac{S_i(t)+m\bar r(t)}{N_i(t)+m},\qquad m=20.$$

Здесь $\bar r(t)$ — глобальная историческая средняя оценка, $S_i(t)$ — сумма оценок товара, $N_i(t)$ — их количество. При маленьком $N_i$ оценка ближе к глобальной средней.

Ниже — полный состав признаков одного snapshot. Везде $w<t$ означает завершённые недели, а окно давности — $[t-4\text{ недели},t)$.

| Признак | Что означает | Как считать |
|---|---|---|
| `user_events_total` | вся история пользователя | $\sum_{w<t} n\_events$ |
| `user_items_unique` | сколько разных товаров оценил пользователь | число уникальных `item_id`, а не сумма недельных `n_unique` |
| `user_mean_rating` | средняя оценка пользователя | $S_u(t)/N_u(t)$ |
| `user_high_rating_share` | доля оценок 4–5 | $H_u(t)/N_u(t)$ |
| `user_events_4w` | недавняя активность | $\sum_{w\in[t-4w,\,t)} n\_events$ |
| `user_days_since_last_event` | дней с последнего события | $(t-\max last\_event\_ts)/86400$ |
| `item_events_total` | вся история товара | $\sum_{w<t} n\_events$ |
| `item_users_unique` | сколько разных пользователей у товара | число уникальных `user_id`, а не сумма недельных `n_unique` |
| `item_mean_rating` | средняя оценка товара | $S_i(t)/N_i(t)$ |
| `item_bayesian_mean_rating` | сглаженная средняя оценка | $(S_i+m\bar r)/(N_i+m)$, $m=20$ |
| `item_events_4w` | недавняя популярность | $\sum_{w\in[t-4w,\,t)} n\_events$ |
| `item_age_weeks` | возраст товара в неделях | $(t-\min first\_event\_ts)/7$ дней |
| `global_events_total` | размер глобальной истории | $\sum_{w<t} rating\_count$ |
| `global_mean_rating` | глобальная средняя оценка | $\sum rating\_sum / \sum rating\_count$ |
| `user_category_events_total` | история пользователя внутри категории | $\sum_{w<t} n\_events$ по товарам категории |
| `user_category_events_4w` | недавняя активность в категории | то же по окну $[t-4w,\,t)$ |
| `user_category_mean_rating` | средняя оценка пользователя в категории | $S_{u,c}/N_{u,c}$ |
| `category_events_total` | популярность категории | $\sum_{w<t} n\_events$ по категории |
| `category_events_4w` | недавняя популярность категории | то же по окну $[t-4w,\,t)$ |

Кросс-признаки (`user_item_rating_diff`, `item_recent_popularity_ratio`, `item_popularity_share`, `user_category_share`, `user_category_preference_lift`) считаются уже после присоединения snapshot к парам `session–item` в §5.

Соберём snapshot признаков в функции `calculate_feature_snapshot(datamart_dir, week_start, lookback_weeks=4)`: прочитайте нужные срезы Aggs через `read_weekly_counters` — всю историю строго до $t$ для lifetime-агрегатов и последние `lookback_weeks` недель для rolling, — затем посчитайте все признаки из таблицы выше. Функция возвращает словарь таблиц: обязательные ключи `"user"`, `"item"`, `"global"`, а при наличии категорий — ещё `"user_category"` и `"category"`. Названия колонок внутри таблиц должны совпадать с таблицей выше. При `lookback_weeks <= 0` — `ValueError`. Категорийный словарь функция должна читать сама из `common/item_categories.parquet`, если он существует.

In [ ]:
def calculate_feature_snapshot(
    datamart_dir: Path,
    week_start: datetime,
    lookback_weeks: int = 4,
) -> dict[str, pl.DataFrame]:
    """Строит point-in-time таблицы признаков на начало недели."""
    if lookback_weeks <= 0:
        raise ValueError("lookback_weeks должен быть положительным")

    # PUT YOUR CODE HERE
    return


tests.check_lifetime_features(
    add_weekly_sessions, build_raw_layer, build_aggs_layer, calculate_feature_snapshot
)
tests.check_rolling_features(
    add_weekly_sessions, build_raw_layer, build_aggs_layer, calculate_feature_snapshot
)
tests.check_point_in_time_leakage(
    add_weekly_sessions, build_raw_layer, build_aggs_layer, calculate_feature_snapshot
)
tests.check_category_features(
    add_weekly_sessions, build_raw_layer, build_aggs_layer, calculate_feature_snapshot
)

Таблица, сохранённая под именем `features/user/week=t.parquet`, означает **признаки, доступные на начало недели** $t$. Это важное соглашение: при сборке обучающих примеров мы не будем дополнительно сдвигать файл на неделю назад.

Подготовку snapshot вынесем в `build_features_layer(datamart_dir, weeks)`: для каждой запрошенной недели постройте snapshot через `calculate_feature_snapshot` и запишите его таблицы в `features/<entity>/week=t.parquet`. Функция опирается на уже сохранённый слой Aggs.

In [ ]:
def build_features_layer(
    datamart_dir: Path, weeks: Sequence[datetime]
) -> None:
    """Строит и сохраняет point-in-time snapshots для заданных недель."""
    # PUT YOUR CODE HERE


tests.check_feature_storage(
    add_weekly_sessions,
    build_raw_layer,
    build_aggs_layer,
    build_features_layer,
    calculate_feature_snapshot,
)


Применим наши функции ко всем взаимодействиям. Исходные события распределятся по Raw, недельные счётчики — по Aggs, а для каждой недели появится набор исторических файлов в Features. В оценочных неделях мы также строим snapshot на их начало.

In [ ]:
# В первый запуск материализуем датамарт полностью. Raw → Aggs → Features:
build_raw_layer(interactions, DATAMART_DIR)
build_aggs_layer(DATAMART_DIR)
prepare_item_categories(items, DATAMART_DIR)

all_weeks = interactions["week_start"].unique().sort().to_list()
build_features_layer(DATAMART_DIR, all_weeks)

print("Недельных snapshots:", len(all_weeks))
print("Первый snapshot:", all_weeks[0])

first_raw_path = sorted(
    (DATAMART_DIR / "raw" / "events").glob("week=*/events.parquet")
)[0]
print("Пример raw-партиции:", first_raw_path, pl.read_parquet(first_raw_path).shape)

for entity in ("user", "item", "global"):
    print(
        entity,
        pl.read_parquet(
            DATAMART_DIR
            / "features"
            / entity
            / f"week={all_weeks[-1].date().isoformat()}.parquet"
        ).shape,
    )

## 5. Обогащение признаками (1p)

В этом разделе реализуем и проверим функцию, которая присоединяет к парам `session–item` рассчитанные ранее признаки. К полным выборкам train, validation и test применим её позже, после формирования retrieval candidate pools.

Обогащение выполняется как point-in-time join: для строки недели $t$ читается snapshot недели $t$, а ключами сущностей служат `user_id`, `item_id` и при необходимости категория товара. Состав входной таблицы при этом не меняется: строки нельзя терять или размножать, а target нельзя использовать при расчёте признаков.

Кросс-признаки вычисляются уже после объединения:

| Признак | Что означает | Как считать |
|---|---|---|
| `user_item_rating_diff` | разлад вкуса пользователя и оценки товара | $user\_mean\_rating_u(t)-item\_mean\_rating_i(t)$ |
| `item_recent_popularity_ratio` | насколько товар популярен сейчас против всей истории | $item\_events\_4w_i(t)/\max(item\_events\_total_i(t),1)$ |
| `item_popularity_share` | доля товара в глобальной истории | $item\_events\_total_i(t)/\max(global\_events\_total(t),1)$ |
| `user_category_share` | какая доля активности пользователя приходится на категорию товара | $user\_category\_events\_total_{u,c}(t)/\max(user\_events\_total_u(t),1)$ |
| `user_category_preference_lift` | во сколько раз интерес к категории выше её доли в истории | $user\_category\_share_{u,c}/\max(category\_events\_total_c(t)/\max(global\_events\_total(t),1),10^{-9})$ |

Для не встречавшихся в истории сущностей нужны согласованные значения по умолчанию: средние и счётчики — 0, `user_days_since_last_event` — большое число, заметно больше любого реального срока (например, 36500).

Для сборки итоговой таблицы понадобится `join_feature_snapshot`: присоедините по нужной неделе user-, item-, global- и при наличии категориальные признаки, затем вычислите кросс-признаки. Не размножайте строки и не используйте текущий рейтинг как входной признак.

In [ ]:
def join_feature_snapshot(pairs: pl.DataFrame, datamart_dir: Path) -> pl.DataFrame:
    """Добавляет к парам session–item признаки на начало их недели."""
    if pairs.is_empty():
        return pairs

    # PUT YOUR CODE HERE
    return pairs


tests.check_feature_join(
    add_weekly_sessions,
    build_raw_layer,
    build_aggs_layer,
    build_features_layer,
    join_feature_snapshot,
)

В CatBoost входят 15 базовых point-in-time признаков датамарта, 7 категорийных и признаки retrieval-этапа: позиция товара у каждой модели, флаг источника и число источников.

In [ ]:
HISTORICAL_FEATURE_COLUMNS = [
    "user_events_total",
    "user_items_unique",
    "user_mean_rating",
    "user_high_rating_share",
    "user_events_4w",
    "user_days_since_last_event",
    "item_events_total",
    "item_users_unique",
    "item_mean_rating",
    "item_bayesian_mean_rating",
    "item_events_4w",
    "item_age_weeks",
    "user_item_rating_diff",
    "item_recent_popularity_ratio",
    "item_popularity_share",
]
CATEGORY_FEATURE_COLUMNS = [
    "user_category_events_total",
    "user_category_events_4w",
    "user_category_mean_rating",
    "category_events_total",
    "category_events_4w",
    "user_category_share",
    "user_category_preference_lift",
]
RETRIEVAL_FEATURE_COLUMNS = [
    "rank_TopPopular",
    "rank_ContentItemKNN",
    "rank_ALS",
    "from_TopPopular",
    "from_ContentItemKNN",
    "from_ALS",
    "source_count",
]


def get_feature_columns(frame: pl.DataFrame) -> list[str]:
    """Проверяет наличие всех признаков и возвращает их в заданном порядке."""
    present = [column for column in CATEGORY_FEATURE_COLUMNS if column in frame.columns]
    if present and len(present) != len(CATEGORY_FEATURE_COLUMNS):
        missing = sorted(set(CATEGORY_FEATURE_COLUMNS) - set(present))
        raise ValueError(f"Категорийные признаки собраны неполностью: {missing}")

    features = (
        HISTORICAL_FEATURE_COLUMNS
        + (CATEGORY_FEATURE_COLUMNS if present else [])
        + RETRIEVAL_FEATURE_COLUMNS
    )
    missing = set(features) - set(frame.columns)

    if missing:
        raise ValueError(f"Отсутствуют признаки: {sorted(missing)}")

    return features

Пока retrieval-признаков нет, продемонстрируем entity join на нескольких
validation-позитивах. Та же функция ниже будет применена к candidate pools.

In [ ]:
example_pairs = val_truth.head(5)
example_features = join_feature_snapshot(example_pairs, DATAMART_DIR)
display(
    example_features.select(
        "session_id",
        "item_id",
        "relevance",
        *HISTORICAL_FEATURE_COLUMNS[:7],
    )
)

## 6. Метрики качества ранжирования (2p)

В предыдущем ДЗ нас интересовало, какие товары попали в пул кандидатов. Теперь важно ещё и в каком порядке они оказались в первых 20 позициях.

`HitRate` показывает, нашёлся ли в top-20 хотя бы один релевантный товар, `MRR` учитывает позицию первого попадания, а `ExpNDCG` оценивает весь порядок с учётом оценок 1–5. Поэтому основной метрикой выбора ranking-модели будет `ExpNDCG@20`, а остальные две помогут интерпретировать результат.

Будем считать `ExpNDCG@20`, `HitRate@20` и `MRR@20`. Все результаты усредняются по недельным сессиям, а не по уникальным пользователям. Для одной сессии может быть несколько релевантных товаров с разными оценками.

### Exponential NDCG@K

Пусть $r_j$ — степень релевантности товара на позиции $j$. Для экспоненциальной версии DCG вклад товара равен $2^{r_j}-1$, а скидка за далёкую позицию задаётся логарифмом:

$$DCG@K=\sum_{j=1}^{K}\frac{2^{r_j}-1}{\log_2(j+1)},\qquad NDCG@K=\frac{DCG@K}{IDCG@K}.$$

`IDCG` — максимум DCG при идеальном расположении **всех релевантных товаров сессии**, а не только товаров, найденных кандидатогенераторами. Например, разница между рейтингами 5 и 1 после экспоненциального преобразования — $31$ против $1$: потерять высокооценённый товар особенно неприятно.

Посчитаем метрику в функции `exp_ndcg_at_k`, используя полный `ground_truth` и проверяя список рекомендаций на дубли. При нулевом `IDCG` возвращаем ноль.

In [ ]:
def _validate_ranking_lists(recommendations: Sequence[str], k: int) -> None:
    """Проверяет значение `k` и отсутствие дублей в одной выдаче."""
    if k <= 0:
        raise ValueError("k должен быть положительным")
    if len(recommendations) != len(set(recommendations)):
        raise ValueError("Внутри одной выдачи не допускаются дубли")


def exp_ndcg_at_k(
    recommendations: Sequence[str], ground_truth: Mapping[str, float], k: int = 20
) -> float:
    """Вычисляет ExpNDCG@k относительно полного ground truth сессии."""
    _validate_ranking_lists(recommendations, k)
    # PUT YOUR CODE HERE
    return 0.0


tests.check_exp_ndcg_at_k(exp_ndcg_at_k)


### HitRate@K

`HitRate` отвечает на простой вопрос: нашёлся ли хотя бы один наблюдаемый товар в первых $K$ рекомендациях?

$$HitRate@K_s=\mathbb{1}[\exists j\le K:r_{sj}>0].$$

Здесь $\mathbb{1}$ — индикатор условия. Итоговая метрика — среднее по сессиям. Она не отличает одну полезную рекомендацию от нескольких и не учитывает величину рейтинга.

Напишите функцию `hit_rate_at_k`.

In [ ]:
def hit_rate_at_k(
    recommendations: Sequence[str], ground_truth: Mapping[str, float], k: int = 20
) -> float:
    """Возвращает 1, если в первых `k` позициях есть релевантный товар."""
    _validate_ranking_lists(recommendations, k)
    # PUT YOUR CODE HERE
    return 0.0


tests.check_hit_rate_at_k(hit_rate_at_k)


### MRR@K

`MRR` обращает внимание только на **первое попадание**. Если первый релевантный товар находится на позиции $q\le K$, то

$$RR@K_s=\frac1q.$$

При отсутствии попаданий $RR@K_s=0$, а $MRR@K$ — среднее значение по сессиям. Например, первая релевантная рекомендация на третьем месте даёт $1/3$.

Аналогично необходимо написать функцию `mrr_at_k`.

In [ ]:
def mrr_at_k(
    recommendations: Sequence[str], ground_truth: Mapping[str, float], k: int = 20
) -> float:
    """Возвращает обратный ранг первого релевантного товара в top-k."""
    _validate_ranking_lists(recommendations, k)
    # PUT YOUR CODE HERE
    return 0.0


tests.check_mrr_at_k(mrr_at_k)


### Общий расчет метрик

Функция `evaluate_ranking` принимает таблицу рекомендаций с `session_id`, `item_id` и `position`, а также полную таблицу ground truth с `relevance`. Для каждой сессии нужно восстановить порядок выдачи и вычислить три метрики. Сессии без попаданий нельзя терять из знаменателя.

Объединим расчёт метрик в функции `evaluate_ranking`, возвращающей словарь из трёх значений с суффиксом `@K`.

In [ ]:
def evaluate_ranking(
    recommendations: pl.DataFrame,
    ground_truth: pl.DataFrame,
    k: int = 20,
) -> dict[str, float]:
    """Вычисляет и макроусредняет ranking-метрики по сессиям."""
    if k <= 0:
        raise ValueError("k должен быть положительным")

    truth_rows = (
        ground_truth.select("session_id", "item_id", "relevance")
        .group_by("session_id")
        .agg(pl.col("item_id"), pl.col("relevance"))
    )
    if truth_rows.is_empty():
        raise ValueError("Пустое множество оценочных сессий")

    # PUT YOUR CODE HERE
    return {}


tests.check_evaluate_ranking(evaluate_ranking)

## 7. Retrieval-кандидаты и hard negatives (2p)

Retrieval-модели сокращают весь каталог до небольшого candidate pool. На следующем этапе CatBoost сможет только переставить найденные товары, поэтому релевантный товар, пропущенный всеми кандидатогенераторами, уже нельзя вернуть в выдачу.

Чтобы увеличить покрытие, объединим TopPopular, ALS и ContentItemKNN: модели используют разные сигналы и находят разные товары. Кандидаты, которых нет в ground truth сессии, станут hard negatives.

Можно было бы добавить в обучение случайные популярные товары, но многие из них не попали бы в candidate pool для конкретного пользователя и никогда не встретились бы ранкеру на inference. Кроме того, такие примеры часто оказываются слишком простыми: модель учится отделять положительный товар от заведомо неподходящего, хотя в реальной выдаче ей нужно выбирать между несколькими правдоподобными кандидатами.

Candidate-based hard negatives уже получили высокий ранг хотя бы у одного retrieval-источника, а значит, похожи на реальные альтернативы по популярности, контенту или истории взаимодействий. Ошибки на них полезнее для обучения финального порядка. Популярные товары при этом не исключаются: они входят в pool через TopPopular, но используются только в тех же условиях, в которых ранкер увидит их при применении. Как и раньше, это псевдонегативы: отсутствие товара в ground truth не означает, что он действительно не нравится пользователю.

Для обучения ранкера выделим
последние 10% train-событий как `ranker-train`; retrieval-модели для этого окна
обучаются только на более ранних 90% train. Для validation retrieval-модели обучаются на полном train, для test — на
`train + validation`. Внутри каждого окна модели фиксированы, а seen-фильтр и
point-in-time признаки обновляются по неделям.

### 7.1. Retrieval-модели из ДЗ 2 (1p)

Перенесите из ДЗ 2 `top_n_indices`, базовые классы `BaseRecommender` и `MatrixRecommender` и три кандидатогенератора: `TopPopularRetriever`, `ALSRetriever` и `TextItemKNNRetriever`.

In [ ]:
def top_n_indices(scores: NDArray[np.float32], n: int) -> NDArray[np.int64]:
    # PUT YOUR CODE HERE
    return


`BaseRecommender` хранит каталог, историю просмотренных товаров и общие процедуры `fit`/`recommend`. Конкретный кандидатогенератор реализует собственные `_fit` и `_recommend`.

Переиспользуем структуру предыдущего домашнего задания, чтобы все три модели возвращали одинаковый формат рекомендаций.

In [ ]:
class BaseRecommender(ABC):
    """Общий интерфейс рекомендательной модели."""

    name: str
    fitted: bool
    items: list[str]
    item_to_index: dict[str, int]
    popular_items: list[str]
    seen_items: dict[str, set[str]]
    show_progress: bool
    batch_size: int = 1024

    def __init__(self, name: str, show_progress: bool = True) -> None:
        self.name = name
        self.show_progress = show_progress
        self.fitted = False

    def fit(self, interactions: pl.DataFrame, **kwargs: Any) -> Self:
        """Запоминает каталог и историю, затем обучает конкретную модель."""
        popularity = (
            interactions.group_by("item_id")
            .agg(pl.len().alias("n_events"))
            .sort(["n_events", "item_id"], descending=[True, False])
        )
        # Каталог сразу хранится по убыванию популярности: это и перечень товаров,
        # и готовая резервная выдача для пользователей без истории.
        self.items = popularity["item_id"].to_list()
        self.item_to_index = {item: index for index, item in enumerate(self.items)}
        self.popular_items = self.items

        seen = (
            interactions.select(["user_id", "item_id"])
            .unique(maintain_order=True)
            .group_by("user_id", maintain_order=True)
            .agg(pl.col("item_id"))
        )
        self.seen_items = {
            user: set(item_list)
            for user, item_list in zip(
                seen["user_id"].to_list(), seen["item_id"].to_list()
            )
        }
        self._fit(interactions, **kwargs)
        self.fitted = True
        return self

    @abstractmethod
    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        """Обучает конкретную модель."""

    def recommend(self, user_ids: Sequence[str], topn: int = 50) -> list[list[str]]:
        """Возвращает списки рекомендаций в порядке user_ids."""
        if not self.fitted:
            raise ValueError("Сначала вызовите fit")
        if topn <= 0:
            raise ValueError("topn должен быть положительным")
        recommendations = self._recommend([str(user) for user in user_ids], topn)
        if len(recommendations) != len(user_ids):
            raise RuntimeError("Число списков рекомендаций не совпало с числом пользователей")
        return recommendations

    @abstractmethod
    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Строит рекомендации конкретной модели."""

    def _remove_seen(
        self,
        user_id: str,
        candidates: Iterable[str],
        topn: int,
    ) -> list[str]:
        """Берёт первые topn кандидатов, пропуская товары из истории пользователя."""
        # PUT YOUR CODE HERE
        return

    def _seen_indices(
        self,
        user_id: str,
        index: Mapping[str, int] | None = None,
    ) -> NDArray[np.int64]:
        """Номера просмотренных товаров в item_to_index или в переданном отображении."""
        # PUT YOUR CODE HERE
        return

    def _rank_by_scores(
        self,
        scores: NDArray[np.float32],
        vocabulary: Sequence[str],
        topn: int,
    ) -> list[list[str]]:
        """Переводит матрицу оценок в списки товаров; скрытые (-inf) отбрасываются.
        Испольузет top_n_indices."""
        # PUT YOUR CODE HERE
        return

    def _recommend_in_batches(
        self,
        user_ids: list[str],
        topn: int,
        is_known: Callable[[str], bool],
        score_known: Callable[[list[str], int], list[list[str]]],
    ) -> list[list[str]]:
        """Общий батч-цикл инференса: пользователи режутся на батчи размера
        batch_size, для известных вызывается score_known, для остальных —
        резервная выдача по popular_items. Этим циклом пользуются Random-,
        Matrix- и Transformer-модели, чтобы не повторять его реализацию трижды."""
        result: list[list[str]] = []
        starts = range(0, len(user_ids), self.batch_size)
        for start in tqdm(starts, desc=self.name, disable=not self.show_progress):
            batch = user_ids[start : start + self.batch_size]
            known = [user for user in batch if is_known(user)]
            ranked = iter(score_known(known, topn))
            for user in batch:
                result.append(
                    next(ranked)
                    if is_known(user)
                    else self._remove_seen(user, self.popular_items, topn)
                )
        return result


`MatrixRecommender` создаёт разреженную матрицу взаимодействий пользователей и товаров. Она используется ALS и контентной моделью.

In [ ]:
class MatrixRecommender(BaseRecommender):
    """Модели поверх разреженной матрицы взаимодействий."""

    users: list[str]
    user_to_id: dict[str, int]
    item_to_id: dict[str, int]
    id_to_item: list[str]
    interaction_matrix: sparse.csr_matrix

    def _create_interaction_matrix(
        self,
        interactions: pl.DataFrame,
        weight_column: str | None = None,
    ) -> None:
        """Строит CSR-матрицу взаимодействий и отображения идентификаторов."""
        self.users = interactions["user_id"].unique(maintain_order=True).to_list()
        self.user_to_id = ... # PUT YOUR CODE HERE
        self.id_to_item = ... # PUT YOUR CODE HERE
        self.item_to_id = ... # PUT YOUR CODE HERE

        # Векторизованный перевод id -> индекс: join с маленькой таблицей-словарём
        # вместо поэлементного .map(dict) по всему датасету.
        user_vocab = pl.DataFrame(
            {"user_id": self.users, "user_index": np.arange(len(self.users))}
        )
        item_vocab = pl.DataFrame(
            {"item_id": self.id_to_item, "item_index": np.arange(len(self.id_to_item))}
        )
        indexed = interactions.join(user_vocab, on="user_id", how="left").join(
            item_vocab, on="item_id", how="left"
        )
        rows = indexed["user_index"].to_numpy()
        columns = indexed["item_index"].to_numpy()
        values = (
            np.ones(len(interactions), dtype=np.float32)
            if weight_column is None
            else indexed[weight_column].to_numpy().astype(np.float32)
        )
        matrix = sparse.csr_matrix(
            (values, (rows, columns)),
            shape=(len(self.users), len(self.id_to_item)),
        )
        if weight_column is None:
            matrix.data[:] = 1.0
        self.interaction_matrix = matrix

    @abstractmethod
    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """Возвращает оценки всех товаров матрицы для переданных пользователей."""

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Батчами считает оценки; холодные пользователи получают популярные товары."""
        return self._recommend_in_batches(
            user_ids, topn, lambda user: user in self.user_to_id, self._rank_known
        )

    def _rank_known(self, user_ids: list[str], topn: int) -> list[list[str]]:
        """Считает оценки для известных пользователей и отбирает top-K."""
        if not user_ids:
            return []
        user_indices = np.array(
            [self.user_to_id[user] for user in user_ids], dtype=np.int64
        )
        scores = np.asarray(self._score_batch(user_indices), dtype=np.float32)
        for row, user in enumerate(user_ids):
            scores[row, self._seen_indices(user, self.item_to_id)] = -np.inf
        return self._rank_by_scores(scores, self.id_to_item, topn)


`TopPopular` — неперсонализированный базовый алгоритм: товары упорядочиваются по частоте исторических событий (или, при другом режиме, по сумме оценок).

In [ ]:
class TopPopularRetriever(BaseRecommender):
    """Сортирует товары по числу событий или по сумме оценок."""

    weighting: Literal["count", "rating"]
    ranked_items: list[str]

    def __init__(
        self,
        weighting: Literal["count", "rating"] = "count",
        show_progress: bool = True,
    ) -> None:
        super().__init__("TopPopular", show_progress)
        self.weighting = weighting

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _recommend(self, user_ids: list[str], topn: int) -> list[list[str]]:
        # PUT YOUR CODE HERE
        return


ALS разлагает матрицу взаимодействий на пользовательские и товарные факторы. Для пары $(u,i)$ оценка имеет вид

$$s_{ui}=x_u^\top y_i,$$

где $x_u$ и $y_i$ — латентные представления пользователя и товара. Используем ту же обёртку над `implicit`, что в предыдущем ДЗ.

In [ ]:
class ALSRetriever(MatrixRecommender):
    """Обёртка над implicit ALS."""

    factors: int
    regularization: float
    alpha: float
    iterations: int
    seed: int
    model: Any

    def __init__(
        self,
        factors: int = 64,
        regularization: float = 0.01,
        alpha: float = 1.0,
        iterations: int = 15,
        seed: int = 42,
        show_progress: bool = True,
    ) -> None:
        super().__init__("ALS", show_progress)
        self.factors = factors
        self.regularization = regularization
        self.alpha = alpha
        self.iterations = iterations
        self.seed = seed

    def _fit(self, interactions: pl.DataFrame, **kwargs: Any) -> None:
        # PUT YOUR CODE HERE
        return

    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """s_ui = x_u^T y_i."""
        # PUT YOUR CODE HERE
        return


Контентная модель кодирует названия товаров в векторы, а затем сравнивает их с профилем пользователя. Векторы нормируем по евклидовой норме, чтобы скалярное произведение соответствовало косинусной близости.

In [ ]:
def encode_titles(
    titles: Sequence[str],
    model_name: str,
    encoder: Any | None = None,
    show_progress: bool = True,
) -> NDArray[np.float32]:
    """Кодирует названия товаров и нормирует векторы."""
    text_encoder = encoder or SentenceTransformer(model_name)
    # PUT YOUR CODE HERE
    return


Загрузка языковой модели и вычисление эмбеддингов может занимать время, поэтому сохраняем векторы названий на диск и используем один и тот же кэш на validation и test.

In [ ]:
class CachedTitleEncoder:
    """Адаптер для получения заранее рассчитанных текстовых эмбеддингов."""

    def __init__(self, encoded: Mapping[str, np.ndarray]):
        self.encoded = encoded

    def encode(self, texts: Sequence[str], **kwargs: Any) -> np.ndarray:
        """Собирает матрицу кэшированных эмбеддингов в порядке `texts`."""
        return np.vstack([self.encoded[str(t)] for t in texts]).astype(np.float32)


def build_cached_title_encoder(
    items: pl.DataFrame,
    model_name: str = "sentence-transformers/paraphrase-MiniLM-L3-v2",
    cache_dir: Path = DATA_DIR / "ranking_cache",
) -> CachedTitleEncoder:
    """Загружает эмбеддинги названий из кэша или вычисляет и сохраняет их."""
    unique_titles = sorted(set(map(str, items["title"].to_list())) | {"unknown title"})
    signature = hashlib.sha256(
        (model_name + "\n" + "\n".join(unique_titles)).encode("utf-8")
    ).hexdigest()[:16]
    cache_path = cache_dir / f"titles_{signature}.npz"

    if cache_path.exists():
        with np.load(cache_path, allow_pickle=False) as data:
            vectors = data["vectors"]
    else:
        encoder = SentenceTransformer(model_name)
        vectors = encode_titles(
            unique_titles, model_name, encoder=encoder, show_progress=True
        )
        cache_dir.mkdir(parents=True, exist_ok=True)
        np.savez_compressed(cache_path, vectors=vectors)

    return CachedTitleEncoder(dict(zip(unique_titles, vectors)))

В этом ДЗ названием `ContentItemKNN` обозначена контентная модель из прошлого ДЗ. Для пользователя она строит профиль по названиям уже оценённых товаров и ранжирует остальные товары по близости к профилю. Это **не** коллаборативный ItemKNN на матрице совместных встречаемостей.

In [ ]:
class TextItemKNNRetriever(MatrixRecommender):
    """Сравнивает профиль пользователя с векторами названий товаров."""

    weighting: Literal["uniform", "rating"]
    model_name: str
    encoder: Any | None
    item_embeddings: NDArray[np.float32]
    user_profiles: NDArray[np.float32]

    def __init__(
        self,
        weighting: Literal["uniform", "rating"] = "rating",
        model_name: str = "sentence-transformers/paraphrase-MiniLM-L3-v2",
        encoder: Any | None = None,
        show_progress: bool = True,
    ) -> None:
        super().__init__("ContentItemKNN", show_progress)
        self.weighting = weighting
        self.model_name = model_name
        self.encoder = encoder

    def _fit(
        self,
        interactions: pl.DataFrame,
        *,
        items: pl.DataFrame,
        **kwargs: Any,
    ) -> None:
        # PUT YOUR CODE HERE
        return

    def _score_batch(self, user_indices: NDArray[np.int64]) -> NDArray[np.float32]:
        """s_ui = cos(p_u, e_i) — оба вектора уже нормированы."""
        # PUT YOUR CODE HERE
        return


Кандидатные списки строит готовая функция `generate_candidate_lists`: для всех оценочных сессий она берёт до 100 рекомендаций от каждого генератора, удаляет товары из истории пользователя до начала недели и сохраняет позиции в исходном порядке.

In [ ]:
def generate_candidate_lists(
    models: Mapping[str, Any],
    evaluation_sessions: pl.DataFrame,
    history: pl.DataFrame,
    topn: int = 100,
) -> dict[str, pl.DataFrame]:
    """Строит недельные списки кандидатов для каждой retrieval-модели."""
    if topn <= 0:
        raise ValueError("topn должен быть > 0")

    results: dict[str, list[tuple[str, datetime, str, str, int]]] = {
        name: [] for name in models
    }

    weeks = (
        evaluation_sessions.select("week_start")
        .unique()
        .sort("week_start")["week_start"]
        .to_list()
    )

    for week in weeks:
        session_rows = evaluation_sessions.filter(pl.col("week_start") == week).sort(
            "session_id"
        )
        users = session_rows["user_id"].to_list()
        sessions = session_rows["session_id"].to_list()

        past = (
            history.filter(pl.col("timestamp") < week)
            .select("user_id", "item_id")
            .unique()
        )
        past_map = {
            u: set(items)
            for u, items in past.group_by("user_id").agg("item_id").iter_rows()
        }

        for name, model in models.items():
            old_map = model.seen_items
            n_extra = max(
                (len(past_map.get(u, set()) - old_map.get(u, set())) for u in users),
                default=0,
            )
            request_n = min(len(model.items), topn + n_extra)
            raw_recs = (
                model.recommend(users, topn=request_n)
                if request_n
                else [[] for _ in users]
            )

            for session_id, user, items in zip(sessions, users, raw_recs):
                kept: list[str] = []
                seen = past_map.get(user, set())

                for item in items:
                    if item not in seen and item not in kept:
                        kept.append(item)
                    if len(kept) == topn:
                        break

                results[name].extend(
                    (session_id, week, user, item, position)
                    for position, item in enumerate(kept, start=1)
                )

    schema = {
        "session_id": pl.String,
        "week_start": pl.Datetime("us", "UTC"),
        "user_id": pl.String,
        "item_id": pl.String,
        "position": pl.Int32,
    }

    return {
        name: pl.DataFrame(rows, schema=schema, orient="row")
        for name, rows in results.items()
    }

### 7.2. Объединение и разметка кандидатов (1p)

Реализуйте `merge_candidates`: объедините источники по `(session_id, week_start, user_id, item_id)` (колонки `week_start` и `user_id` должны остаться в результате), сохраните позицию каждого источника в `rank_<source>`, добавьте флаги `from_<source>` и число источников `source_count`. Отсутствующая позиция получает `RETRIEVAL_K + 1`; после дедупликации один товар должен встречаться в сессии ровно один раз.

Реализуйте `label_candidates`: присоедините полный ground truth недели и заполните ненаблюдавшиеся пары значениями `relevance=0`, `binary_target=0`. Затем реализуйте `filter_informative_candidate_groups`: оставьте для обучения только сессии минимум с двумя кандидатами и хотя бы одним положительным примером.

In [ ]:
def merge_candidates(
    recommendations_by_model: Mapping[str, pl.DataFrame],
    retrieval_k: int = RETRIEVAL_K,
) -> pl.DataFrame:
    """Объединяет retrieval-выдачи и добавляет признаки источников."""
    if not recommendations_by_model:
        raise ValueError("Нет кандидатогенераторов")

    # PUT YOUR CODE HERE
    return pl.DataFrame()


tests.check_candidate_deduplication(merge_candidates)

In [ ]:
def label_candidates(
    candidates: pl.DataFrame, ground_truth: pl.DataFrame
) -> pl.DataFrame:
    """Добавляет кандидатам `relevance` и бинарную целевую переменную."""
    # PUT YOUR CODE HERE
    return candidates


tests.check_candidate_labels(label_candidates)


In [ ]:
def filter_informative_candidate_groups(frame: pl.DataFrame) -> pl.DataFrame:
    """Оставляет группы с двумя кандидатами и хотя бы одним положительным."""
    # PUT YOUR CODE HERE
    return frame


tests.check_informative_candidate_groups(filter_informative_candidate_groups)


Внутренний split нужен только для согласованного обучения второго этапа. Сессии,
в candidate pool которых не попал ни один positive item, исключаются из обучения
ранкера: изменить их порядок недостаточно, сначала нужно улучшать retrieval.

Для validation кандидатогенераторы обучаются на `train` и остаются фиксированными на всём отложенном периоде. При этом фильтр просмотренного и датамарт для каждой сессии учитывают события только до начала соответствующей недели. Validation pool строится один раз и остаётся неизменным для обоих Optuna-поисков.

In [ ]:
# Эмбеддинги названий всех товаров создаём / загружаем ровно один раз:
title_encoder = build_cached_title_encoder(items)

In [ ]:
def make_retrievers() -> dict[str, BaseRecommender]:
    """Создаёт необученные TopPopular, ContentItemKNN и ALS."""
    return {
        "TopPopular": TopPopularRetriever(weighting="count", show_progress=False),
        "ContentItemKNN": TextItemKNNRetriever(
            weighting="rating", encoder=title_encoder, show_progress=False
        ),
        "ALS": ALSRetriever(
            factors=32,
            regularization=0.01,
            alpha=1.0,
            iterations=15,
            seed=SEED,
            show_progress=False,
        ),
    }


def fit_retrievers(history: pl.DataFrame) -> dict[str, BaseRecommender]:
    """Обучает три retrieval-модели на одной исторической выборке."""
    models = make_retrievers()

    models["TopPopular"].fit(history)
    models["ContentItemKNN"].fit(history, items=items)
    models["ALS"].fit(history)

    return models

In [ ]:
retrieval_train_events, ranker_part_a, ranker_part_b = temporal_week_split(
    train_events, 0.9, 0.05, 0.05
)

ranker_train_events = pl.concat([ranker_part_a, ranker_part_b], how="vertical_relaxed")
ranker_weeks = ranker_train_events.select("week_start").unique()
ranker_truth = positives.join(ranker_weeks, on="week_start", how="semi")
ranker_sessions = ranker_truth.select("session_id", "week_start", "user_id").unique()

ranker_retrievers = fit_retrievers(retrieval_train_events)
ranker_lists = generate_candidate_lists(
    ranker_retrievers,
    ranker_sessions,
    interactions,
    RETRIEVAL_K,
)
ranker_union = merge_candidates(ranker_lists)
ranker_labeled = label_candidates(ranker_union, ranker_truth)
train_candidate_basis = filter_informative_candidate_groups(ranker_labeled)
train_rank = join_feature_snapshot(train_candidate_basis, DATAMART_DIR)

train_retrievers = fit_retrievers(train_events)
val_sessions = val_truth.select("session_id", "week_start", "user_id").unique()
validation_lists = generate_candidate_lists(
    train_retrievers,
    val_sessions,
    interactions,
    RETRIEVAL_K,
)
val_union = merge_candidates(validation_lists)
val_candidate_features = join_feature_snapshot(val_union, DATAMART_DIR)
val_labeled = label_candidates(val_union, val_truth)
val_candidate_basis = filter_informative_candidate_groups(val_labeled)
val_rank = join_feature_snapshot(val_candidate_basis, DATAMART_DIR)

FEATURE_COLUMNS = get_feature_columns(train_rank)
assert FEATURE_COLUMNS == get_feature_columns(val_candidate_features)

print("Ranker-train events:", len(ranker_train_events))
print(
    "Ranker-train candidate groups:",
    train_rank["session_id"].n_unique(),
    "rows:",
    len(train_rank),
)
print("Validation unique candidates:", val_union.shape)
print("Features:", FEATURE_COLUMNS)

display(
    train_rank.group_by("binary_target")
    .agg(pl.len().alias("rows"))
    .sort("binary_target")
)

## 8. Ranking dataset и общий scoring (2p)

После объединения retrieval-выдач одна строка датасета соответствует уникальной паре `(session_id, item_id)`. В ней находятся target, point-in-time признаки и retrieval-признаки: позиции товара, флаги источников и число источников.

Для `CatBoostRanker` все строки одной `session_id` образуют query group; классификатор получает те же строки без группового loss. Скоринг и построение top-20 вынесем в общие функции, чтобы обе модели различались только способом обучения.

Реализуйте `filter_informative_ranking_groups`: оставьте только сессии минимум с двумя кандидатами и двумя различающимися значениями `relevance`. Затем реализуйте `prepare_ranking_pool`: отсортируйте строки так, чтобы каждая `session_id` образовывала непрерывную query group, и передайте в `Pool` признаки, `relevance` и идентификаторы групп.

In [ ]:
def filter_informative_ranking_groups(frame: pl.DataFrame) -> pl.DataFrame:
    """Оставляет группы, содержащие не менее двух уровней relevance."""
    # PUT YOUR CODE HERE
    return frame


tests.check_informative_ranking_groups(filter_informative_ranking_groups)


In [ ]:
def prepare_ranking_pool(frame: pl.DataFrame, features: Sequence[str]) -> Pool:
    """Создаёт CatBoost Pool с соседними строками одной ranking-группы."""
    ordered = frame.sort(["session_id", "item_id"])

    if not set(features).issubset(ordered.columns):
        raise ValueError("Отсутствуют некоторые признаки")

    # PUT YOUR CODE HERE
    return Pool()


tests.check_ranking_groups(prepare_ranking_pool)

Скоринг вынесем в `score_candidates`: получите матрицу признаков в том же порядке, что при обучении; для `classifier` верните вероятность положительного класса, для `ranker` — значение `predict`. При неизвестном `model_type` выбрасывается `ValueError`. Функция обязана возвращать по одному конечному вещественному числу на строку.

In [ ]:
def score_candidates(
    model: Any, features: pl.DataFrame, model_type: str, feature_columns: Sequence[str]
) -> np.ndarray:
    """Возвращает оценку каждого кандидата для выбранного типа CatBoost."""
    scores = ...  # PUT YOUR CODE HERE

    scores = np.asarray(scores, dtype=np.float64).reshape(-1)
    if len(scores) != len(features) or not np.isfinite(scores).all():
        raise ValueError("Некорректная длина или значения predictions")

    return scores


tests.check_candidate_scoring(score_candidates)


Для финальной перестановки напишите `rerank_candidates`: проверьте, что на входе нет дублей, посчитайте по одному score на товар, отсортируйте внутри сессии по убыванию score и выберите top-20. При равенстве скоров сортируйте по `item_id`, чтобы результат не зависел от случайного порядка строк.

In [ ]:
def rank_candidates_by_score(
    candidates: pl.DataFrame,
    score_column: str,
    k: int = EVAL_K,
) -> pl.DataFrame:
    """Сортирует кандидатов по готовой оценке и возвращает первые `k`."""
    if score_column not in candidates.columns:
        raise ValueError(f"Нет score-колонки {score_column}")

    return (
        candidates.sort(
            ["session_id", score_column, "item_id"],
            descending=[False, True, False],
        )
        .with_columns((pl.int_range(pl.len()).over("session_id") + 1).alias("position"))
        .filter(pl.col("position") <= k)
    )


def rerank_candidates(
    model: Any,
    candidates: pl.DataFrame,
    model_type: str,
    feature_columns: Sequence[str],
    k: int = 20,
) -> pl.DataFrame:
    """Оценивает уникальных кандидатов и возвращает первые `k` каждой сессии."""
    if k <= 0:
        raise ValueError("k должен быть положительным")

    if candidates.select(
        pl.struct(["session_id", "item_id"]).is_duplicated().any()
    ).item():
        raise ValueError("Перед scoring удалите дубли кандидатов")

    # PUT YOUR CODE HERE
    return candidates


tests.check_rerank_candidates(rerank_candidates)

Classifier и Ranker получают одинаковые признаки и одинаковые candidate groups. Различаются только target и loss. Следующая ячейка готовит входы двух моделей из уже собранного датасета.

In [ ]:
X_train = train_rank.select(FEATURE_COLUMNS).to_numpy().astype(np.float32)
y_binary = train_rank["binary_target"].to_numpy()

ranker_train_rank = filter_informative_ranking_groups(train_rank)
train_group_pool = prepare_ranking_pool(
    ranker_train_rank,
    FEATURE_COLUMNS,
)

print("Classifier rows:", len(X_train))
print("YetiRank groups:", ranker_train_rank["session_id"].n_unique())

## 9. CatBoostClassifier + Optuna (1p)

Первый вариант второго этапа — **pointwise**-классификация кандидатов. `CatBoostClassifier` предсказывает, была ли строка наблюдаемым взаимодействием (`binary_target=1`) или candidate-based псевдонегативом (`binary_target=0`). `Logloss` оптимизируется по отдельным строкам и не сравнивает товары одной сессии напрямую.

Чтобы превратить классификатор в ранкер, упорядочиваем товары одной сессии по вероятности положительного класса. Эта вероятность зависит от стратегии сэмплирования негативов и не обязана быть откалиброванной вероятностью клика.

Подберём `iterations`, `depth`, `learning_rate`, `l2_leaf_reg` и `random_strength` с помощью Optuna. Для воспроизводимости используем `TPESampler(seed=42)` и 10 trials на полном датасете, без сокращения данных.

Пространство параметров задаёт готовая функция `suggest_catboost_params`, возвращающая конфигурацию одного trial. Конструкторы обеих моделей также приведены ниже: они фиксируют loss, seed и служебные параметры. Один и тот же диапазон используем для двух постановок, чтобы сравнение было понятным.

In [ ]:
def suggest_catboost_params(trial: optuna.Trial) -> dict[str, Any]:
    """Предлагает набор гиперпараметров CatBoost для одного Optuna trial."""
    return {
        "iterations": trial.suggest_int("iterations", 200, 600, step=100),
        "depth": trial.suggest_int("depth", 4, 8),
        "learning_rate": trial.suggest_float("learning_rate", 0.02, 0.2, log=True),
        "l2_leaf_reg": trial.suggest_float("l2_leaf_reg", 1.0, 20.0, log=True),
        "random_strength": trial.suggest_float("random_strength", 0.0, 2.0),
    }


def make_classifier(params: Mapping[str, Any]) -> CatBoostClassifier:
    """Создаёт CatBoostClassifier с общей конфигурацией эксперимента."""
    return CatBoostClassifier(
        loss_function="Logloss",
        auto_class_weights="Balanced",
        random_seed=SEED,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
        **params,
    )


def make_ranker(params: Mapping[str, Any]) -> CatBoostRanker:
    """Создаёт CatBoostRanker с общей конфигурацией эксперимента."""
    return CatBoostRanker(
        loss_function="YetiRank",
        random_seed=SEED,
        allow_writing_files=False,
        verbose=False,
        thread_count=-1,
        **params,
    )

Не будем выбирать ранкер по ROC-AUC на псевдонегативах: нам важнее качество настоящей рекомендательной выдачи. Для каждого trial обучаем CatBoost на `train_rank`, скорим фиксированный `val_candidate_features`, выбираем top-20 в каждой сессии и считаем `ExpNDCG@20` относительно полного validation ground truth.

Реализуйте `classifier_objective`, максимизирующую validation ExpNDCG@20.

In [ ]:
def classifier_objective(trial: optuna.Trial) -> float:
    """Обучает classifier и возвращает validation ExpNDCG@20."""
    # PUT YOUR CODE HERE
    return 0.0


После реализации objective готовая ячейка запускает Optuna и обучает `CatBoostClassifier` на `train_rank` с лучшими параметрами. До выбора типа ранкера ещё не заглядываем в `test`.

In [ ]:
classifier_study = optuna.create_study(
    direction="maximize", sampler=TPESampler(seed=SEED)
)
classifier_study.optimize(
    classifier_objective, n_trials=10, show_progress_bar=True
)

classifier_params = classifier_study.best_params

classifier_model = make_classifier(classifier_params)

classifier_model.fit(X_train, y_binary)

print("Classifier best validation ExpNDCG:", classifier_study.best_value)
print("Classifier params:", classifier_params)

## 10. CatBoostRanker + Optuna (1p)

Второй вариант — `CatBoostRanker` с функцией потерь `YetiRank`. Ранкер получает `session_id` как query group и обучается сравнивать кандидатов одной выдачи; target задаётся исходной `relevance` от 0 до 5.

Ранкер возвращает непрерывные скоры, поэтому при инференсе используем `model.predict(X)`. Значения скоров сравниваются внутри одного запроса; трактовать их как вероятности не нужно.

Оптимизируем те же гиперпараметры и по той же внешней метрике ExpNDCG@20, что и классификатор. Внутри CatBoost алгоритм обучается с `YetiRank`, а Optuna оценивает качество top-20 на одинаковых отложенных кандидатах.

Для ранкера потребуется отдельная `ranker_objective`. Сравнивать Logloss и YetiRank loss напрямую нельзя: масштабы и цели у них разные.

In [ ]:
def ranker_objective(trial: optuna.Trial) -> float:
    """Обучает YetiRank и возвращает validation ExpNDCG@20."""
    # PUT YOUR CODE HERE
    return 0.0


После реализации objective готовая ячейка запускает 10 trials, выбирает лучшие параметры и обучает `CatBoostRanker` на тех же обучающих неделях.

In [ ]:
ranker_study = optuna.create_study(direction="maximize", sampler=TPESampler(seed=SEED))
ranker_study.optimize(ranker_objective, n_trials=10, show_progress_bar=True)

ranker_params = ranker_study.best_params

ranker_model = make_ranker(ranker_params)

ranker_model.fit(train_group_pool)

print("Ranker best validation ExpNDCG:", ranker_study.best_value)
print("Ranker params:", ranker_params)

## 11. Важность признаков (1p)

После двух Optuna-поисков проверим не только лучшие значения метрики. История trials покажет разброс `ExpNDCG@20` между конфигурациями, а feature importance — какие исторические и retrieval-признаки использует каждая модель.

Поскольку модели решают разные задачи, полезно сравнить, на какие признаки они опираются. Feature importance измеряет влияние признака внутри конкретной обученной модели, но не является оценкой причинного эффекта.

Готовая ячейка изображает validation ExpNDCG@20 в порядке запуска trial и служит проверкой обеих objective-функций. Не стоит интерпретировать график как монотонную кривую обучения: разные trials соответствуют разным конфигурациям гиперпараметров.

In [ ]:
_, axes = plt.subplots(1, 2, figsize=(13, 4))

for study, name, ax in [
    (classifier_study, "Classifier", axes[0]),
    (ranker_study, "Ranker", axes[1]),
]:
    vals = [t.value for t in study.trials if t.value is not None]
    ax.plot(range(1, len(vals) + 1), vals, marker="o")
    ax.set(title=f"Optuna: {name}", xlabel="Trial", ylabel="ExpNDCG@20")

plt.tight_layout()
plt.show()

Визуализацию важностей вынесем в `plot_feature_importance`: получите важности признаков CatBoost, отсортируйте их, покажите top-15 и верните полную таблицу. Для `CatBoostRanker` при необходимости передайте training Pool.

Обратите внимание, какие признаки связаны с исторической активностью пользователя, какие описывают популярность товара, а какие — их сочетание. Высокая корреляция признаков может перераспределять важности между ними.

In [ ]:
def plot_feature_importance(model, pool: Pool | None, title: str) -> pl.DataFrame:
    """Рисует top-15 важностей и возвращает полную таблицу признаков."""
    # PUT YOUR CODE HERE
    return pl.DataFrame()


In [ ]:
classifier_importances = plot_feature_importance(
    classifier_model, None, "CatBoostClassifier: top-15"
)
ranker_importances = plot_feature_importance(
    ranker_model, train_group_pool, "CatBoostRanker: top-15"
)
display(classifier_importances.head(10))
display(ranker_importances.head(10))

## 12. Выбор ranking-модели по validation

Classifier и Ranker сравним на одном зафиксированном validation candidate pool. Состав кандидатов, признаки и функция внешней оценки одинаковы, поэтому различие метрик относится к выбранной постановке и её гиперпараметрам.

Победителя выберем по `ExpNDCG@20`, а при равенстве — по `MRR@20`. После этого тип модели и параметры фиксируются; `test` в выборе не участвует.

In [ ]:
validation_reports = {}
ranking_models = {
    "Union + CatBoostClassifier": (classifier_model, "classifier"),
    "Union + CatBoostRanker": (ranker_model, "ranker"),
}

for name, (model, model_type) in ranking_models.items():
    ranked = rerank_candidates(
        model,
        val_candidate_features,
        model_type,
        FEATURE_COLUMNS,
        EVAL_K,
    )
    validation_reports[name] = evaluate_ranking(ranked, val_truth, EVAL_K)

validation_comparison = pl.DataFrame(
    [
        {"Pipeline": name, **report}
        for name, report in validation_reports.items()
    ]
).sort(f"ExpNDCG@{EVAL_K}", descending=True)
display(validation_comparison)

winner = sorted(
    ranking_models,
    key=lambda name: (
        -validation_reports[name][f"ExpNDCG@{EVAL_K}"],
        -validation_reports[name][f"MRR@{EVAL_K}"],
        name,
    ),
)[0]
best_ranker_type = ranking_models[winner][1]
best_ranker_params = (
    classifier_params if best_ranker_type == "classifier" else ranker_params
)

print("Лучший ranking pipeline:", winner)

plt.figure(figsize=(10, 4))
plt.bar(
    validation_comparison["Pipeline"],
    validation_comparison[f"ExpNDCG@{EVAL_K}"],
)
plt.ylabel(f"ExpNDCG@{EVAL_K}")
plt.title("Validation: выбор CatBoost на union candidate pool")
plt.tight_layout()
plt.show()

## 13. Финальный эксперимент на `test`

После фиксации конфигурации переобучим retrieval-модели и выбранный CatBoost на `train + validation`. Затем один раз оценим готовый пайплайн на `test`; события тестовых недель не используются ни для переобучения, ни для пересмотра параметров. 

Сравнение построим так, чтобы разделить изменение порядка и изменение состава кандидатов. Пара `retrieval` / `retrieval + CatBoost` использует один candidate pool и показывает эффект reranking. `Union + CatBoost` работает с объединённым пулом, поэтому его результат дополнительно зависит от кандидатов, найденных другими источниками.

В итоговую таблицу войдут исходные TopPopular, ContentItemKNN и ALS, каждый одиночный источник после CatBoost и CatBoost на объединённом пуле кандидатов.

Тип модели и гиперпараметры уже зафиксированы на validation. Теперь добавляем
информативные validation candidate groups в ranking train и переобучаем выбранный
CatBoost.

In [ ]:
train_val_rank = pl.concat(
    [train_rank, val_rank],
    how="vertical_relaxed",
)
train_val_events = pl.concat(
    [train_events, val_events],
    how="vertical_relaxed",
)

if best_ranker_type == "classifier":
    final_ranker = make_classifier(best_ranker_params)

    final_ranker.fit(
        train_val_rank.select(FEATURE_COLUMNS).to_numpy().astype(np.float32),
        train_val_rank["binary_target"].to_numpy(),
    )
else:
    final_ranker = make_ranker(best_ranker_params)

    final_ranker.fit(
        prepare_ranking_pool(
            filter_informative_ranking_groups(train_val_rank),
            FEATURE_COLUMNS,
        )
    )

Обучим три retrieval-модели на `train + validation` и зафиксируем их на всём test-периоде. Для пользователей тестовых недель генерируем по 100 рекомендаций с исключением уже просмотренных на начало недели; point-in-time признаки также учитывают только завершённые недели. Такой порядок согласован с подготовкой validation-кандидатов.

In [ ]:
final_retrievers = fit_retrievers(train_val_events)

test_sessions = test_truth.select("session_id", "week_start", "user_id").unique()
test_lists = generate_candidate_lists(
    final_retrievers,
    test_sessions,
    interactions,
    RETRIEVAL_K,
)

test_union = merge_candidates(test_lists)
unique_test_features = join_feature_snapshot(test_union, DATAMART_DIR)

Чтобы оценить собственное качество каждого кандидатогенератора, оставим его исходные первые 20 рекомендаций. 

In [ ]:
result_rows = []

for name, recs in test_lists.items():
    report = evaluate_ranking(
        recs.filter(pl.col("position") <= EVAL_K),
        test_truth,
        EVAL_K,
    )
    result_rows.append({"Pipeline": name, **report})

Теперь для каждого генератора сохраняем то же множество top-100 товаров, но сортируем их с помощью выбранного CatBoost. Это сравнение измеряет эффект именно изменения порядка, а не расширения пула.

In [ ]:
for name in test_lists:
    source_candidates = unique_test_features.filter(pl.col(f"from_{name}") == 1)

    new_order = rerank_candidates(
        final_ranker,
        source_candidates,
        best_ranker_type,
        FEATURE_COLUMNS,
        EVAL_K,
    )

    report = evaluate_ranking(new_order, test_truth, EVAL_K)
    result_rows.append({"Pipeline": f"{name} + CatBoost", **report})

В последнем эксперименте берём кандидатов **от всех трёх моделей**. Один товар может встретиться в нескольких списках, поэтому применяем следующий порядок:

$$\text{TopPopular}\cup\text{ContentItemKNN}\cup\text{ALS}\;\to\;\text{unique}(session,item)\;\to\;\text{features}\;\to\;\text{CatBoost}\;\to\;\text{top-20}.$$

Дедупликация происходит до скоринга, не после. `Union + CatBoost` отличается от одиночных источников и составом кандидатов, поэтому его выигрыш нельзя приписывать только ранкеру.

In [ ]:
union_ranked = rerank_candidates(
    final_ranker,
    unique_test_features,
    best_ranker_type,
    FEATURE_COLUMNS,
    EVAL_K,
)
result_rows.append(
    {
        "Pipeline": "Union + CatBoost",
        **evaluate_ranking(union_ranked, test_truth, EVAL_K),
    }
)

TEST_METRICS = pl.DataFrame(result_rows).sort(
    f"ExpNDCG@{EVAL_K}",
    descending=True,
)
display(TEST_METRICS)

Соберём таблицу `ExpNDCG@20`, `HitRate@20`, `MRR@20` и сравним исходные и переупорядоченные выдачи. Для каждого отдельного источника полезно рассчитать **разницу** `ExpNDCG` после и до CatBoost.

In [ ]:
labels = TEST_METRICS["Pipeline"].to_list()
values = TEST_METRICS[f"ExpNDCG@{EVAL_K}"].to_list()

plt.figure(figsize=(12, 4))
plt.bar(range(len(labels)), values)
plt.xticks(range(len(labels)), labels, rotation=35, ha="right")
plt.ylabel(f"ExpNDCG@{EVAL_K}")
plt.title("Test: retrieval baselines и ranking")
plt.tight_layout()
plt.show()

for name in ("TopPopular", "ContentItemKNN", "ALS"):
    base = TEST_METRICS.filter(pl.col("Pipeline") == name)[f"ExpNDCG@{EVAL_K}"][0]
    reranked = TEST_METRICS.filter(pl.col("Pipeline") == f"{name} + CatBoost")[
        f"ExpNDCG@{EVAL_K}"
    ][0]

    print(f"{name}: delta ExpNDCG@{EVAL_K} = {reranked - base:+.5f}")

best_retrieval_test = max(
    ("TopPopular", "ContentItemKNN", "ALS"),
    key=lambda name: TEST_METRICS.filter(pl.col("Pipeline") == name)[
        f"ExpNDCG@{EVAL_K}"
    ][0],
)

best_retrieval_value = TEST_METRICS.filter(pl.col("Pipeline") == best_retrieval_test)[
    f"ExpNDCG@{EVAL_K}"
][0]
union_value = TEST_METRICS.filter(pl.col("Pipeline") == "Union + CatBoost")[
    f"ExpNDCG@{EVAL_K}"
][0]

print(
    "Union + CatBoost vs best retrieval:",
    f"{union_value - best_retrieval_value:+.5f}",
)

## 14. Выводы по домашнему заданию (1p)

В выводах нужно интерпретировать полученные таблицы. Расскажите о том, что удалось пронаблюдать, а также ответьте на следующие вопросы:

1. Почему candidate-based hard negatives лучше согласованы с inference, чем
   случайные популярные товары?
2. Какие retrieval-признаки вошли в top feature importance?
3. Чем отличаются classifier и ranker по target и loss?
4. Какая модель победила на validation?
5. Улучшил ли CatBoost каждый отдельный retrieval source на test?
6. Почему `Union + CatBoost` сравнивается с лучшим одиночным retrieval отдельно?
7. Какие группы исключаются из ranking train и почему ranking не может исправить
   отсутствие positive item в candidate pool?
8. Как point-in-time snapshots предотвращают temporal leakage и согласуется с продовым пайплайном?